# 2×2 Network-on-Chip — Traffic Benchmark

One Vivado simulation runs six independent workloads: LOW / MEDIUM / HIGH × UNIFORM / HOTSPOT. This notebook computes **end-to-end packet latency, throughput, backpressure and traffic matrices** and saves vector SVG plots for both FAST=0 and FAST=1.

**Protocol:** latency starts at the *source accepted header flit* and ends at the *destination accepted tail flit* (3-flit packet). The golden model and FPGA area/timing analysis remain separate deliverables.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, SVG

# Works when Jupyter is launched from repo root or analysis/.
HERE = Path.cwd()
ANALYSIS = HERE / "analysis" if (HERE / "analysis" / "analyze_noc.py").exists() else HERE
sys.path.insert(0, str(ANALYSIS))
from analyze_noc import *

DATA_DIRS = [ANALYSIS / "data" / f"Fast={fast}" for fast in (0, 1)]
OUT_DIR = ANALYSIS / "output"
FIG_DIR = OUT_DIR / "figures"
style()

def preview(name):
    paths = sorted(FIG_DIR.glob(name + ".svg")) + sorted(FIG_DIR.glob("FAST=*/" + name + ".svg"))
    for path in paths:
        print(path.relative_to(FIG_DIR))
        display(SVG(filename=str(path)))

print("Data folders:", [str(path.resolve()) for path in DATA_DIRS])

## 1. Load & validate data

In [ ]:
packets, cycles, summary = load_runs(DATA_DIRS)
metrics = compute_metrics(packets, cycles, summary)
write_summary(metrics, OUT_DIR)
display(metrics.round(4))

## 2. Mean packet latency vs accepted injection

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_latency_vs_injection_rate(metrics, FIG_DIR)
preview("01_mean_latency_vs_injection_rate")

## 3. P95 packet latency vs accepted injection

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_latency_vs_injection_rate(metrics, FIG_DIR, "P95LatencyCycles", "02_p95_latency_vs_injection_rate")
preview("02_p95_latency_vs_injection_rate")

## 4. Accepted input and completed output throughput vs threshold

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_throughput(metrics, FIG_DIR)
preview("03_throughput_vs_generator_threshold")

## 5. Endpoint backpressure vs injection rate

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_stalls(metrics, FIG_DIR)
preview("04_backpressure_vs_injection_rate")

## 6. Latency distributions for every configuration

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_latency_box(packets, FIG_DIR)
preview("05_latency_distribution_boxplots")

## 7. High-rate latency ECDF

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_high_rate_ecdf(packets, FIG_DIR)
preview("06_high_rate_latency_ecdf")

## 8. High-hotspot flit transfer activity

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
for fast in sorted(cycles.FAST.unique()):
    plot_flit_activity(cycles[cycles.FAST == fast], FIG_DIR / f"FAST={int(fast)}")
preview("07_high_hotspot_flit_activity")

## 9. In-flight and buffered packet count

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
for fast in sorted(cycles.FAST.unique()):
    plot_outstanding(cycles[cycles.FAST == fast], FIG_DIR / f"FAST={int(fast)}")
preview("08_high_hotspot_outstanding_packets")

## 10. Source-destination matrix: high-rate uniform

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
for fast in sorted(packets.FAST.unique()):
    plot_matrix(packets[packets.FAST == fast], FIG_DIR / f"FAST={int(fast)}", "uniform")
preview("09_traffic_matrix_high_uniform")

## 11. Source-destination matrix: high-rate hotspot

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
for fast in sorted(packets.FAST.unique()):
    plot_matrix(packets[packets.FAST == fast], FIG_DIR / f"FAST={int(fast)}", "hotspot")
preview("09_traffic_matrix_high_hotspot")

## 12. High-rate delivery by source

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_source_fairness(packets, FIG_DIR)
preview("10_high_rate_deliveries_by_source")

## 13. Latency vs Manhattan hop count

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_hop_latency(packets, FIG_DIR)
preview("11_latency_vs_hop_count")

## 14. High-hotspot cumulative accepted vs delivered

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
for fast in sorted(cycles.FAST.unique()):
    plot_packet_progress(cycles[cycles.FAST == fast], FIG_DIR / f"FAST={int(fast)}")
preview("12_high_hotspot_packet_progress")

## 15. Latency vs configured generator threshold

Each plot is written as **SVG (vector)** to `analysis/output/figures/`.

In [ ]:
plot_grouped_rate_latency(metrics, FIG_DIR)
preview("13_latency_vs_generator_threshold")

## Golden-model-style comparison

Uniform and hotspot panels compare baseline and fast variants. Latency uses packets delivered during measurement, matching `model/benchmark.py`; earlier plots use measurement-injected packets including drain deliveries. The x-axis is the RTL generator threshold, not the model's offered flit rate. Generation-to-tail (queued) latency is unavailable because the RTL CSVs have no generation timestamps. These are performance plots, not a cycle-level equivalence check.

In [ ]:
plot_model_comparison(metrics, FIG_DIR)
preview("mean_latency")
preview("delivered_flits_node_cycle")

### Interpretation cautions

- The source generator's hexadecimal threshold is **not** an offered packet rate. Use measured accepted packet injection on the primary latency-vs-injection plot.
- Warmup packets can complete in the measurement window. The primary latency plots use measurement-**injected** packets, including drain deliveries; the model-style latency plot uses measurement-**delivered** packets.
- Endpoint stalls are not the same as internal router credit starvation. Full congestion attribution needs router-internal counters.
- A zero-loss test does not formally prove fairness or deadlock freedom.
- To prove paper-vs-baseline area/Fmax you must independently synthesize and implement both RTL variants.